# E3 - Uncertainty-Driven Distillation

E3 keeps the E2 distilled perturbation setup, fixes beta at `0.75`, and changes only which rollout states receive the 2% CF-label budget.

Arms: `uniform`, `uncertainty`, and `active` (`uncertainty_leverage`). The cluster sweep uses Taxi, DoorKey-6x6, UnlockPickup, and RedBlueDoors-6x6 with seeds `0, 1, 2`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from config import make_config
from scripts.train import run_seed


## Choose a local check

Use a small frame budget locally. The cluster pipeline uses the YAML budgets through `./submit_e3.sh`.

In [ ]:
ENV_CONFIGS = ('taxi_cf', 'doorkey6x6_cf', 'unlockpickup_cf', 'redbluedoors6x6_cf')
ENV_CONFIG = 'doorkey6x6_cf'
SEEDS = (0,)
FRAMES = 100_000
LABEL_FRACTION = 0.02
BETA = 0.75

ARMS = {
    'uniform': 'uniform',
    'uncertainty': 'uncertainty',
    'active': 'uncertainty_leverage',
}


In [ ]:
def e3_config(arm: str, seed: int):
    if arm not in ARMS:
        raise KeyError(f'unknown arm: {arm}')
    overrides = {
        'ppo.pg_mode': 'landscape_distill',
        'ppo.cf_subsample': LABEL_FRACTION,
        'ppo.norm_adv': 'batch',
        'run.run_name': f'local_e3_{ENV_CONFIG}_{arm}_b{BETA:g}',
        'run.seeds': (seed,),
        'run.record_trajectories': False,
        'run.log_every_updates': 1,
        'distill.beta': BETA,
        'distill.query_strategy': ARMS[arm],
    }
    if FRAMES is not None:
        overrides['ppo.total_timesteps'] = FRAMES
    return make_config(ENV_CONFIG, **overrides)

print(e3_config('active', SEEDS[0]).summary())


## Run paired E3 arms

In [ ]:
artifacts = {}
for arm in ARMS:
    for seed in SEEDS:
        print(f'\n=== E3 {ENV_CONFIG}: {arm}, seed {seed}, beta {BETA:g} ===')
        artifacts[(arm, seed)] = run_seed(e3_config(arm, seed), seed, progress=True)

artifacts


## Inspect E3 diagnostics

`cf_query_*` columns describe the selected teacher-query states. `distill_query_*` columns describe the rollout-wide uncertainty and leverage before querying. `distill_heldout_*` columns audit selected fresh labels before they are added to replay.

In [ ]:
import pandas as pd

for (arm, seed), item in artifacts.items():
    scalars = pd.read_csv(item['scalars_csv'])
    cols = [c for c in scalars.columns if c.startswith(('cf_query_', 'distill_query_', 'distill_heldout_', 'perturb_'))]
    print(f'\n{arm}, seed {seed}')
    display(scalars[['global_step', 'success_rate_100', *cols]].tail())
